In [1]:
!pip install scikit-learn

In [2]:
import sys
!{sys.executable} -m pip install xgboost

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef
)

import xgboost as xgb


In [4]:
columns = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race",
    "sex", "capital-gain", "capital-loss", "hours-per-week",
    "native-country", "income"
]

train_df = pd.read_csv("/home/cloud/Downloads/adult.data", names=columns, sep=",", skipinitialspace=True)

print(train_df.shape)

(32561, 15)


In [5]:

# Handle missing values
train_df = train_df.replace("?", np.nan)
train_df = train_df.dropna()


In [6]:
test_df = pd.read_csv("/home/cloud/Downloads/adult.test", names=columns, sep=",", skiprows=1, skipinitialspace=True)

# Handle missing values
test_df = test_df.replace("?", np.nan)
test_df = test_df.dropna()

# Remove period from income column
test_df["income"] = test_df["income"].str.replace(".", "", regex=False)


In [7]:
train_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [8]:
test_df.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K


In [9]:
#Encode categorical variable

le = LabelEncoder()

for column in train_df.columns:
    if train_df[column].dtype == 'object':
        train_df[column] = le.fit_transform(train_df[column])


for column in test_df.columns:
    if test_df[column].dtype == 'object':
        test_df[column] = le.fit_transform(test_df[column])


In [10]:
#train,test split

X_train = train_df.drop("income", axis=1)
y_train = train_df["income"]

X_test = test_df.drop("income", axis=1)
y_test = test_df["income"]

In [11]:
print(X_train.shape)
print(y_train.shape)

(30162, 14)
(30162,)


In [12]:
print(X_test.shape)
print(y_test.shape)

(15060, 14)
(15060,)


In [13]:
#Train model
#Logistic Regression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred1 = lr.predict(X_test)
y_prob1 = lr.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred1)
precision = precision_score(y_test, y_pred1)
recall = recall_score(y_test, y_pred1)
f1 = f1_score(y_test, y_pred1)
auc = roc_auc_score(y_test, y_prob1)
mcc = matthews_corrcoef(y_test, y_pred1)

print("Logistic Regression Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

Logistic Regression Metrics
Accuracy : 0.800597609561753
Precision: 0.6504100129477773
Recall   : 0.4072972972972973
F1 Score : 0.5009140767824497
AUC      : 0.8100209959078798
MCC      : 0.4008882361301372


/home/cloud/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [14]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
y_pred2 = dt.predict(X_test)
y_prob2 = dt.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred2)
precision = precision_score(y_test, y_pred2)
recall = recall_score(y_test, y_pred2)
f1 = f1_score(y_test, y_pred2)
auc = roc_auc_score(y_test, y_prob2)
mcc = matthews_corrcoef(y_test, y_pred2)

print("Decision Tree Classifier Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

Decision Tree Classifier Metrics
Accuracy : 0.801062416998672
Precision: 0.5921465968586388
Recall   : 0.6113513513513513
F1 Score : 0.6015957446808511
AUC      : 0.737218333650552
MCC      : 0.46918045576249867


In [15]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred3 = knn.predict(X_test)
y_prob3 = knn.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred3)
precision = precision_score(y_test, y_pred3)
recall = recall_score(y_test, y_pred3)
f1 = f1_score(y_test, y_pred3)
auc = roc_auc_score(y_test, y_prob3)
mcc = matthews_corrcoef(y_test, y_pred3)

print("KNN Classifier Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

KNN Classifier Metrics
Accuracy : 0.7677954847277556
Precision: 0.5461993627674101
Recall   : 0.32432432432432434
F1 Score : 0.4069866033576395
AUC      : 0.6647229491815759
MCC      : 0.2884998663053775


In [16]:
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred4 = nb.predict(X_test)
y_prob4 = nb.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred4)
precision = precision_score(y_test, y_pred4)
recall = recall_score(y_test, y_pred4)
f1 = f1_score(y_test, y_pred4)
auc = roc_auc_score(y_test, y_prob4)
mcc = matthews_corrcoef(y_test, y_pred4)

print("NB Classifier Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

NB Classifier Metrics
Accuracy : 0.7885790172642763
Precision: 0.6469248291571754
Recall   : 0.307027027027027
F1 Score : 0.41642228739002934
AUC      : 0.8221595807955843
MCC      : 0.3386189633063155


In [17]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred5 = nb.predict(X_test)
y_prob5 = nb.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred5)
precision = precision_score(y_test, y_pred5)
recall = recall_score(y_test, y_pred5)
f1 = f1_score(y_test, y_pred5)
auc = roc_auc_score(y_test, y_prob5)
mcc = matthews_corrcoef(y_test, y_pred5)

print("Random Forest Classifier Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

Random Forest Classifier Metrics
Accuracy : 0.7885790172642763
Precision: 0.6469248291571754
Recall   : 0.307027027027027
F1 Score : 0.41642228739002934
AUC      : 0.8221595807955843
MCC      : 0.3386189633063155


In [18]:
xgb_model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)
xgb_model.fit(X_train, y_train)
y_pred6 = nb.predict(X_test)
y_prob6 = nb.predict_proba(X_test)[:,1]

accuracy = accuracy_score(y_test, y_pred6)
precision = precision_score(y_test, y_pred6)
recall = recall_score(y_test, y_pred6)
f1 = f1_score(y_test, y_pred6)
auc = roc_auc_score(y_test, y_prob6)
mcc = matthews_corrcoef(y_test, y_pred6)

print("XG Boost Classifier Metrics")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUC      :", auc)
print("MCC      :", mcc)

/home/cloud/anaconda3/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [19:56:30] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XG Boost Classifier Metrics
Accuracy : 0.7885790172642763
Precision: 0.6469248291571754
Recall   : 0.307027027027027
F1 Score : 0.41642228739002934
AUC      : 0.8221595807955843
MCC      : 0.3386189633063155


In [19]:
import os

os.makedirs("model", exist_ok=True)

In [20]:
import joblib

joblib.dump(lr, "model/logistic.pkl")
joblib.dump(dt, "model/decision_tree.pkl")
joblib.dump(knn, "model/knn.pkl")
joblib.dump(nb, "model/naive_bayes.pkl")
joblib.dump(rf, "model/random_forest.pkl")
joblib.dump(xgb_model, "model/xgboost.pkl")

['model/xgboost.pkl']